In [1]:
import pyspark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1783332221339_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


In [30]:
silver_df = spark.read.parquet(
    "s3a://airline-dataset-2020-2025/Silver/"
)

print("Rows :", silver_df.count())
print("Columns :", len(silver_df.columns))

('Rows :', 40910253)
('Columns :', 120)

In [4]:
print(silver_df.columns)

['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Marketing_Airline_Network', 'Operated_or_Branded_Code_Share_Partners', 'DOT_ID_Marketing_Airline', 'IATA_Code_Marketing_Airline', 'Flight_Number_Marketing_Airline', 'Originally_Scheduled_Code_Share_Airline', 'DOT_ID_Originally_Scheduled_Code_Share_Airline', 'IATA_Code_Originally_Scheduled_Code_Share_Airline', 'Flight_Num_Originally_Scheduled_Code_Share_Airline', 'Operating_Airline', 'DOT_ID_Operating_Airline', 'IATA_Code_Operating_Airline', 'Tail_Number', 'Flight_Number_Operating_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'Tax

In [5]:
eda_df = silver_df.select('FlightDate','CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay','DayOfWeek')

In [6]:
eda_df.show(5)

+-------------------+------------+------------+--------+-------------+-----------------+---------+
|         FlightDate|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|DayOfWeek|
+-------------------+------------+------------+--------+-------------+-----------------+---------+
|2021-07-15 00:00:00|        null|        null|    null|         null|             null|        4|
|2021-07-15 00:00:00|        null|        null|    null|         null|             null|        4|
|2021-07-15 00:00:00|        null|        null|    null|         null|             null|        4|
|2021-07-15 00:00:00|        null|        null|    null|         null|             null|        4|
|2021-07-15 00:00:00|        null|        null|    null|         null|             null|        4|
+-------------------+------------+------------+--------+-------------+-----------------+---------+
only showing top 5 rows

In [7]:
eda_df.summary().show()

+-------+------------------+-----------------+------------------+-------------------+------------------+------------------+
|summary|      CarrierDelay|     WeatherDelay|          NASDelay|      SecurityDelay| LateAircraftDelay|         DayOfWeek|
+-------+------------------+-----------------+------------------+-------------------+------------------+------------------+
|  count|           7644267|          7644267|           7644267|            7644267|           7644267|          40910253|
|   mean|25.263957028188575|4.279215521906809|13.147254144838216|0.13603436405347955| 27.12500452953828|3.9922956966313556|
| stddev| 75.45408450244801|34.04622253656452|31.962116802658198| 3.5225089292055705|61.031506334288096| 2.006259919797299|
|    min|               0.0|              0.0|               0.0|                0.0|               0.0|                 1|
|    25%|               0.0|              0.0|               0.0|                0.0|               0.0|                 2|
|    50%

In [8]:
eda_df.describe()

DataFrame[summary: string, CarrierDelay: string, WeatherDelay: string, NASDelay: string, SecurityDelay: string, LateAircraftDelay: string, DayOfWeek: string]

In [9]:
eda_df.printSchema()

root
 |-- FlightDate: timestamp (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)
 |-- DayOfWeek: integer (nullable = true)

## Cleaning
Here I have changed the datatype of FlightDate from timestamp to Date.

### Why ?
#### Memory Storage Breakdown
- DateType **(4 bytes)**: Spark stores dates internally as a single 32-bit integer representing the number of days since the Unix epoch (1970-01-01). Because it only tracks the day, month, and year, it requires very little space.

- TimestampType **(8 bytes)**: Spark stores timestamps internally as a 64-bit long integer representing microseconds since the Unix epoch. It requires double the memory because it must track hours, minutes, seconds, fractional seconds, and timezone offsets.

In [10]:
from pyspark.sql.functions import col, to_date
df_clean = eda_df.withColumn("FlightDate", to_date(col("FlightDate")))

In [11]:
df_clean.printSchema()

root
 |-- FlightDate: date (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)
 |-- DayOfWeek: integer (nullable = true)

In [12]:
df_clean.show(5)

+----------+------------+------------+--------+-------------+-----------------+---------+
|FlightDate|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|DayOfWeek|
+----------+------------+------------+--------+-------------+-----------------+---------+
|2021-07-15|        null|        null|    null|         null|             null|        4|
|2021-07-15|        null|        null|    null|         null|             null|        4|
|2021-07-15|        null|        null|    null|         null|             null|        4|
|2021-07-15|        null|        null|    null|         null|             null|        4|
|2021-07-15|        null|        null|    null|         null|             null|        4|
+----------+------------+------------+--------+-------------+-----------------+---------+
only showing top 5 rows

In [13]:
for c in df_clean.columns:
    print(c)
    # Get distinct values, limit to 20 so it doesn't flood the screen, and collect
    unique_vals = [row[0] for row in silver_df.select(c).distinct().limit(20).collect()]
    print(unique_vals)
    print("-" * 40)

FlightDate
[datetime.datetime(2021, 7, 20, 0, 0), datetime.datetime(2021, 7, 8, 0, 0), datetime.datetime(2023, 3, 20, 0, 0), datetime.datetime(2021, 8, 27, 0, 0), datetime.datetime(2025, 2, 26, 0, 0), datetime.datetime(2025, 9, 23, 0, 0), datetime.datetime(2024, 7, 31, 0, 0), datetime.datetime(2020, 3, 31, 0, 0), datetime.datetime(2022, 4, 7, 0, 0), datetime.datetime(2024, 10, 13, 0, 0), datetime.datetime(2023, 4, 4, 0, 0), datetime.datetime(2022, 1, 3, 0, 0), datetime.datetime(2024, 9, 27, 0, 0), datetime.datetime(2024, 9, 22, 0, 0), datetime.datetime(2023, 7, 23, 0, 0), datetime.datetime(2020, 7, 13, 0, 0), datetime.datetime(2021, 12, 13, 0, 0), datetime.datetime(2023, 9, 2, 0, 0), datetime.datetime(2025, 11, 2, 0, 0), datetime.datetime(2022, 1, 2, 0, 0)]
----------------------------------------
CarrierDelay
[305.0, 496.0, 299.0, 558.0, 692.0, 934.0, 769.0, 596.0, 1051.0, 2815.0, 1761.0, 147.0, 184.0, 810.0, 170.0, 576.0, 720.0, 782.0, 1369.0, 1765.0]
--------------------------------

#### Converting double to int
##### Why?
- Cuts Size in Half: It reduces the column's memory and storage footprint by 50% (from 8 bytes down to 4 bytes).
- Speeds Up Performance: Less data means faster network transfer (shuffling) between cluster nodes during operations like groupBy.
- Improves Disk Compression: Integers compress much more efficiently than floating-point decimals in columnar formats like Parquet.
- Eliminates Waste: For metrics like flight delays (e.g., 15 minutes), keeping them as doubles wastes space storing meaningless .0 decimals.


In [15]:
delay_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

for c in delay_cols:
    df_clean = df_clean.withColumn(c, col(c).cast("int"))
    
df_clean.printSchema()

root
 |-- FlightDate: date (nullable = true)
 |-- CarrierDelay: integer (nullable = true)
 |-- WeatherDelay: integer (nullable = true)
 |-- NASDelay: integer (nullable = true)
 |-- SecurityDelay: integer (nullable = true)
 |-- LateAircraftDelay: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)

In [22]:
# CarrierDelay_df = df_clean.filter(
#     col('CarrierDelay').isNotNull() & (col('CarrierDelay') > 0)
# ).select('CarrierDelay')

# CarrierDelay_df.summary().show()

In [21]:
from pyspark.sql.functions import col, when

# List of columns you want to analyze
delay_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

# Replace 0 and Null with None for each column dynamically
filtered_df = df_clean.select([
    when((col(c).isNotNull()) & (col(c) > 0), col(c)).alias(c) 
    for c in delay_cols
])

# View summaries side by side
filtered_df.summary().show()

+-------+-----------------+------------------+-----------------+------------------+------------------+
|summary|     CarrierDelay|      WeatherDelay|         NASDelay|     SecurityDelay| LateAircraftDelay|
+-------+-----------------+------------------+-----------------+------------------+------------------+
|  count|          4256432|            462135|          3703330|             38359|           3738361|
|   mean|  45.372375971236| 70.78335551299945|27.13804089832664|27.109231210406946|55.465691248116485|
| stddev|96.50105571333698|120.27591269990215|41.58144075346052| 41.73164410172995| 77.74762764641167|
|    min|                1|                 1|                1|                 1|                 1|
|    25%|                9|                15|                8|                10|                16|
|    50%|               20|                34|               17|                18|                32|
|    75%|               44|                78|               30|         

The above is the summary of each individual columns on the basis of values which are greater than 0.

In [26]:
from pyspark.sql.functions import col, avg, count, when

# Group by DayofWeek and calculate averages/counts
weekday_analysis = df_clean.groupBy("DayofWeek").agg(
    count("*").alias("TotalFlights"),
    # Average delays (ignoring zeros and nulls using 'when')
    avg(when(col("CarrierDelay") > 0, col("CarrierDelay"))).alias("AvgCarrierDelay"),
    avg(when(col("WeatherDelay") > 0, col("WeatherDelay"))).alias("AvgWeatherDelay"),
    avg(when(col("NASDelay") > 0, col("NASDelay"))).alias("AvgNASDelay"),
    avg(when(col("LateAircraftDelay") > 0, col("LateAircraftDelay"))).alias("AvgLateAircraftDelay")
).orderBy("DayofWeek")

# Show the results cleanly
weekday_analysis.show(truncate=False)

+---------+------------+------------------+-----------------+------------------+--------------------+
|DayofWeek|TotalFlights|AvgCarrierDelay   |AvgWeatherDelay  |AvgNASDelay       |AvgLateAircraftDelay|
+---------+------------+------------------+-----------------+------------------+--------------------+
|1        |6105492     |45.912749051806514|72.82182681234727|26.52077707621547 |56.64767589641019   |
|2        |5583713     |46.39101595145158 |73.64738378699673|26.67977941349831 |55.42998257810253   |
|3        |5686712     |45.31314465519569 |71.90417333867094|28.167791547854833|54.392563903950425  |
|4        |6069233     |43.65121108449691 |70.04700048886096|27.93498454062142 |54.11351708469709   |
|5        |6105950     |44.524155925040965|68.54208481256813|26.75231261530424 |54.65909162807177   |
|6        |5327980     |46.98108770851221 |71.22134440503605|26.878056485327527|56.64994853826782   |
|7        |6031173     |45.316080252626385|68.12459237602609|27.00131822402353 |56